# 📊 Retail Sales Dashboard
**Author:** Ahmed Walid  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn  
**Goal:** Analyze retail sales data to uncover trends, top products, and regional performance.


## 1. 📦 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='darkgrid', palette='muted')
print('✅ Libraries loaded successfully')

## 2. 🗄️ Generate Sales Dataset

In [ ]:
np.random.seed(42)
n = 1000

regions    = ['North', 'South', 'East', 'West', 'Central']
categories = ['Electronics', 'Clothing', 'Food & Beverages', 'Home & Garden', 'Sports']
products = {
    'Electronics':      ['Laptop', 'Phone', 'Tablet', 'Headphones', 'Smartwatch'],
    'Clothing':         ['T-Shirt', 'Jeans', 'Jacket', 'Dress', 'Shoes'],
    'Food & Beverages': ['Coffee', 'Tea', 'Snacks', 'Juice', 'Chocolate'],
    'Home & Garden':    ['Sofa', 'Lamp', 'Plant Pot', 'Curtains', 'Rug'],
    'Sports':           ['Yoga Mat', 'Dumbbells', 'Running Shoes', 'Bicycle', 'Tent'],
}
price_range = {
    'Electronics': (200, 1500), 'Clothing': (20, 200),
    'Food & Beverages': (5, 50), 'Home & Garden': (30, 800), 'Sports': (25, 600),
}

cat_col  = np.random.choice(categories, n)
prod_col = [np.random.choice(products[c]) for c in cat_col]
price_col = [round(np.random.uniform(*price_range[c]), 2) for c in cat_col]
qty_col  = np.random.randint(1, 11, n)

df = pd.DataFrame({
    'Order_ID':   [f'ORD-{i:04d}' for i in range(1, n+1)],
    'Date':       pd.date_range('2023-01-01', periods=n, freq='8H')[:n],
    'Region':     np.random.choice(regions, n),
    'Category':   cat_col,
    'Product':    prod_col,
    'Unit_Price': price_col,
    'Quantity':   qty_col,
})
df['Revenue']      = (df['Unit_Price'] * df['Quantity']).round(2)
df['Profit']       = (df['Revenue'] * np.random.uniform(0.15, 0.40, n)).round(2)
df['Profit_Margin']= (df['Profit'] / df['Revenue'] * 100).round(1)
df['Month']        = df['Date'].dt.to_period('M')
df['Month_Name']   = df['Date'].dt.strftime('%b %Y')

print(f'✅ Dataset created: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

## 3. 🔍 Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Info ===')
print(df.dtypes)
print(f'\nMissing values:\n{df.isnull().sum()}')

In [ ]:
print('=== Key Statistics ===')
summary = pd.DataFrame({
    'Total Orders':   [f"{len(df):,}"],
    'Total Revenue':  [f"${df['Revenue'].sum():,.0f}"],
    'Total Profit':   [f"${df['Profit'].sum():,.0f}"],
    'Avg Order Value':[f"${df['Revenue'].mean():,.0f}"],
    'Avg Profit Margin': [f"{df['Profit_Margin'].mean():.1f}%"],
})
summary

## 4. 📈 Visualizations

### 4.1 Monthly Revenue Trend

In [ ]:
monthly = df.groupby('Month')['Revenue'].sum().reset_index()
monthly['Month_str'] = monthly['Month'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(monthly['Month_str'], monthly['Revenue'], alpha=0.15, color='#2196F3')
ax.plot(monthly['Month_str'], monthly['Revenue'], marker='o', color='#2196F3', linewidth=2.5)
ax.set_title('Monthly Revenue Trend', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Month'); ax.set_ylabel('Revenue ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('monthly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: monthly_revenue.png')

### 4.2 Revenue by Category

In [ ]:
cat_rev = df.groupby('Category')['Revenue'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#1565C0','#1976D2','#1E88E5','#42A5F5','#90CAF9']
cat_rev.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Total Revenue by Category', fontsize=16, fontweight='bold', pad=15)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
for i, v in enumerate(cat_rev):
    ax.text(v + 500, i, f'${v:,.0f}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('revenue_by_category.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: revenue_by_category.png')

### 4.3 Regional Performance

In [ ]:
region_data = df.groupby('Region').agg(
    Revenue=('Revenue','sum'),
    Profit=('Profit','sum'),
    Orders=('Order_ID','count')
).reset_index().sort_values('Revenue', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette = sns.color_palette('Blues_d', len(region_data))

# Revenue bar
axes[0].bar(region_data['Region'], region_data['Revenue'], color=palette)
axes[0].set_title('Revenue by Region', fontsize=14, fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))

# Profit bar
axes[1].bar(region_data['Region'], region_data['Profit'], color=palette)
axes[1].set_title('Profit by Region', fontsize=14, fontweight='bold')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('regional_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: regional_performance.png')

### 4.4 Top 10 Products by Revenue

In [ ]:
top10 = df.groupby('Product')['Revenue'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(x=top10.values, y=top10.index, palette='Blues_r', ax=ax)
ax.set_title('Top 10 Products by Revenue', fontsize=16, fontweight='bold', pad=15)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('top10_products.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: top10_products.png')

### 4.5 Profit Margin Heatmap (Region × Category)

In [ ]:
pivot = df.pivot_table(values='Profit_Margin', index='Region', columns='Category', aggfunc='mean')

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='Blues', linewidths=0.5,
            cbar_kws={'label': 'Profit Margin (%)'}, ax=ax)
ax.set_title('Average Profit Margin (%) — Region × Category', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('profit_margin_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: profit_margin_heatmap.png')

## 5. 📌 KPI Summary

In [ ]:
total_revenue = df['Revenue'].sum()
total_profit  = df['Profit'].sum()
avg_margin    = df['Profit_Margin'].mean()
best_region   = df.groupby('Region')['Revenue'].sum().idxmax()
best_category = df.groupby('Category')['Revenue'].sum().idxmax()
best_product  = df.groupby('Product')['Revenue'].sum().idxmax()
best_month    = df.groupby('Month_Name')['Revenue'].sum().idxmax()

print('=' * 45)
print('         📊 RETAIL SALES KPI REPORT')
print('=' * 45)
print(f'  💰 Total Revenue      : ${total_revenue:>12,.0f}')
print(f'  📈 Total Profit       : ${total_profit:>12,.0f}')
print(f'  📊 Avg Profit Margin  : {avg_margin:>11.1f}%')
print(f'  🏆 Best Region        : {best_region:>15}')
print(f'  🏆 Best Category      : {best_category:>15}')
print(f'  🏆 Best Product       : {best_product:>15}')
print(f'  📅 Best Month         : {best_month:>15}')
print('=' * 45)

## 6. 💾 Export Cleaned Data

In [ ]:
df.to_csv('retail_sales_data.csv', index=False)
print(f'✅ Data exported → retail_sales_data.csv ({len(df):,} rows)')

## 7. 📝 Conclusion

### Key Findings:
- **Electronics** consistently drives the highest revenue across all regions.
- **North** and **West** regions outperform others in both revenue and profit.
- Revenue shows clear **seasonal peaks** in Q2 and Q4.
- **Food & Beverages** achieves the highest profit margins despite lower ticket prices.

### Recommendations:
1. Increase inventory of top-performing Electronics products before Q4.
2. Investigate underperforming regions (South/Central) for growth opportunities.
3. Promote high-margin Food & Beverages products via bundling strategies.
